#Fuentes de dato
Fuente  Sales Analysis on Northwind Database

https://www.kaggle.com/datasets/emmanueltugbeh/northwind-orders-and-order-details

Northwind base de datos de ejemplo creada por Microsoft.

Tablas utilizadas: **orders** (pedidos) y **order_details** (línea de pedido), relacionadas por **order_id**.

#Problema y objetivo

##Problema:

Los registros de pedidos de Northwind presentan errores de captura y formato que impiden calcular métricas de ventas confiables.

##Objetivo:

Diagnosticar la calidad de los datos como paso previo a la limpieza y el análisis.

# Extracción incial

##Dimensiones, columnas y tipo de tabla.

In [1]:
import gdown
id_details = "1xgyipusoHkOZEtYKIIJfqhVw1aIsEkIS"
archivo_details = f"https://drive.google.com/uc?id={id_details}"
details = "order_details.csv"

id_orders = "1WumjkcK1WaNtjWcLaLbPQNH5URJ3vSs0"
archivo_orders = f"https://drive.google.com/uc?id={id_orders}"
orders = "orders.csv"

gdown.download(archivo_details, details, quiet=False)
gdown.download(archivo_orders, orders, quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1xgyipusoHkOZEtYKIIJfqhVw1aIsEkIS
To: /content/order_details.csv
100%|██████████| 41.7k/41.7k [00:00<00:00, 46.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1WumjkcK1WaNtjWcLaLbPQNH5URJ3vSs0
To: /content/orders.csv
100%|██████████| 103k/103k [00:00<00:00, 56.2MB/s]


'orders.csv'

In [2]:
import pandas as pd

df_details = pd.read_csv(details, dtype=str, keep_default_na=False)
df_orders = pd.read_csv(orders, dtype=str, keep_default_na=False)

print("Order_details", df_details.shape)
print("Orders", df_orders.shape)
display(df_details.head())
display(df_orders.head())

Order_details (2215, 5)
Orders (865, 14)


,order_id,product_id,unit_price,quantity,discount
0,10474,40,14.7,21,0
1,10527,36,19,30,0.1
2,10731,51,53,30,0.05
3,11009,24,4.5,12,0
4,10657,60,34,30,0


,order_id,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
0,10869,SEVES,5,1998-02-04,1998-03-04,1998-02-09,1,143.28,Seven Seas Imports,90 Wadhurst Rd.,London,,OX15 4NB,UK
1,10440,SAVEA,4,1997-02-10,1997-03-10,1997-02-28,2,86.53,Save-a-lot Markets,187 Suffolk Ln.,Boise,ID,83720,USA
2,10951,RICSU,9,1998-03-16,1998-04-27,1998-04-07,2,30.85,Richter Supermarkt,Starenweg 5,Genève,,1204,Switzerland
3,10876,BONAP,7,1998-02-09,1998-03-09,1998-02-12,3,60.42,Bon app',"12, rue des Bouchers",Marseille,,13008,France
4,11047,EASTC,7,1998-04-24,1998-05-22,1998-05-01,3,46.62,Eastern Connection,35 King George,London,,WX3 6FW,U.K.


#Diagnóstico

##Valores Faltantes

##¿Por que pasar los datos como texto y no en el formato en que estaban?

Al pasar los datos como texto, se evita que pandas adivine los tipos y transforme la información. Por ende se quiere evitar esto para dejar evidencia de los errores del dataset.

Para diagnosticar la calidad de los datos es necesrio ver los valores tal como vienen en el archivo. Este es un primer paso del diagnóstico. Después, en la limpieza, sí se convertirá cada columna a su tipo correcto (númeor, fechas) con reglas claras y documentadas.


In [3]:
normal = pd.read_csv(details)

texto = df_details
print("LECTURA NORMAL")
print("Nulos (NaN) en unit_price:", normal["unit_price"].isna().sum())
print()
print("LECTURA COMO TEXTO")
print("Vacíos:", (texto["unit_price"].str.strip() == "").sum())
print("Valores 'N/A'", (texto["unit_price"] == "N/A").sum())
print("Valores 'free':", (texto["unit_price"] == "free").sum())

LECTURA NORMAL
Nulos (NaN) en unit_price: 33

LECTURA COMO TEXTO
Vacíos: 25
Valores 'N/A' 8
Valores 'free': 6


La lecutra normal reporta 33 valores nulos, pero en realidad son 25 vacíos más 8 **N/A** mezclados. No distingue entre una celda vacía y un **N/A** escrito, y los 6 **free** ni aparecen como problema.

###Conteo de valores faltantes

In [4]:
def faltantes(df):
  vacios = df.apply(lambda col: col.str.strip().eq("").sum())
  resumen = pd.DataFrame({
      "vacios": vacios,
      "porcentaje": (vacios / len(df) * 100).round(2)
  })
  return resumen.sort_values("vacios", ascending=False)


print("order_details")
display(faltantes(df_details))
print()
print("orders")
display(faltantes(df_orders))

order_details


,vacios,porcentaje
discount,31,1.40
unit_price,25,1.13
quantity,20,0.90
product_id,15,0.68
order_id,0,0.00



orders


,vacios,porcentaje
ship_region,525,60.69
freight,27,3.12
shipped_date,23,2.66
ship_postal_code,21,2.43
customer_id,16,1.85
ship_city,15,1.73
ship_name,12,1.39
employee_id,12,1.39
order_date,11,1.27
required_date,11,1.27


Se recorre cada columna del DataFrame, se identifica las celdas que están vacías o contienen únicamente espacios y cuenta cuántas hay en cada columna.

###Registros duplicados

Se revisan en esta sección dos tipos de duplicados: filas idénticas en todas sus columnas y filas que repiten la clave del registro (order_id en orders; order_id + product_id en order_details) aunque difieran en algún otro campo.

In [5]:


exactos = df_orders.duplicated().sum()
por_id = df_orders.duplicated(subset="order_id").sum()

print("orders")
print("Filas totales:", len(df_orders))
print("order_id únicos:", df_orders["order_id"].nunique())
print("Filas idénticas repetidas:", exactos)
print("Filas con order_id repetido (incluye las idénticas):", por_id)
print("Repetidas con order_id igual pero contenido distinto", por_id - exactos)


orders
Filas totales: 865
order_id únicos: 830
Filas idénticas repetidas: 25
Filas con order_id repetido (incluye las idénticas): 35
Repetidas con order_id igual pero contenido distinto 10


In [6]:
sin_extractos = df_orders.drop_duplicates()
conflicto = sin_extractos[sin_extractos.duplicated(subset="order_id", keep=False)]
conflicto = conflicto.sort_values("order_id")

print("Filas en conflicto:", len(conflicto), "| pedidos afectados:", conflicto["order_id"].nunique())
display(conflicto.head(4))

Filas en conflicto: 20 | pedidos afectados: 10


,order_id,customer_id,employee_id,order_date,required_date,shipped_date,ship_via,freight,ship_name,ship_address,ship_city,ship_region,ship_postal_code,ship_country
620,10263,ERNSH,9,1996-07-23,1996-08-20,1996-07-31,3,146.06,ERNST HANDEL,Kirchgasse 6,Graz,,8010,Austria
452,10263,ERNSH,9,1996-07-23,1996-08-20,1996-07-31,3,146.06,Ernst Handel,Kirchgasse 6,Graz,,8010,Austria
127,10349,SPLIR,7,1996-11-08,1996-12-06,1996-11-15,1,8.63,Split Rail Beer & Ale,P.O. Box 555,LaÃ¼nder,WY,82520,USA
728,10349,SPLIR,7,1996-11-08,1996-12-06,1996-11-15,1,8.63,SPLIT RAIL BEER & ALE,P.O. Box 555,LaÃ¼nder,WY,82520,USA


In [7]:
display(conflicto[["order_id", "customer_id", "order_date", "ship_name"]])

,order_id,customer_id,order_date,ship_name
620,10263,ERNSH,1996-07-23,ERNST HANDEL
452,10263,ERNSH,1996-07-23,Ernst Handel
127,10349,SPLIR,1996-11-08,Split Rail Beer & Ale
728,10349,SPLIR,1996-11-08,SPLIT RAIL BEER & ALE
36,10412,WARTH,1997-01-13,WARTIAN HERKKU
71,10412,WARTH,1997-01-13,Wartian Herkku
55,10457,KOENE,1997-02-25,KÖNIGLICH ESSEN
82,10457,KOENE,1997-02-25,Königlich Essen
260,10552,HILAA,1997-05-29,HILARION-Abastos
515,10552,HILAA,1997-05-29,HILARION-ABASTOS


In [8]:
difieren = conflicto.groupby("order_id").agg(lambda s: s.nunique() > 1).any()
print("Columnas que difieren entre las copias:", difieren[difieren].index.tolist())
mismo_nombre = conflicto.groupby("order_id")["ship_name"].apply(lambda s: s.str.lower().nunique() == 1)
print("Pedidos donde ship_name solo cambia en mayúsculas:", mismo_nombre.sum(), "de", len(mismo_nombre))

Columnas que difieren entre las copias: ['ship_name']
Pedidos donde ship_name solo cambia en mayúsculas: 10 de 10


In [9]:
exactos_d = df_details.duplicated().sum()
por_clave = df_details.duplicated(subset=["order_id", "product_id"]).sum()
sin_exactos_d = df_details.drop_duplicates()
por_clave_tras = sin_exactos_d.duplicated(subset=["order_id", "product_id"]).sum()

print("order_details")
print("Filas totales:", len(df_details))
print("Filas idénticas repetidas:", exactos_d)
print("Repetidas por order_id + product_id:", por_clave)
print("Repetidas por clave tras quitar las idénticas:", por_clave_tras)

order_details
Filas totales: 2215
Filas idénticas repetidas: 60
Repetidas por order_id + product_id: 60
Repetidas por clave tras quitar las idénticas: 0


**Conclusion:** En orders se encontraron 25 filas identicas repetidas y 10 pedidos registrados dos veces con contenido distinto; en todos esos casos la unica diferencia esta en ship_name, escrito con mayusculas distintas.

En order_details se encontraron 60 filas identicas repetidas y ninun conflicto por clave. Tras eliminar los duplicados exactos, quedarian 840 filas en orders (todavia con 10 pedidos repetidos por el formato de ship_name) y 2.155 en order_details.

###Tipos de dato incorrectos
Según el diccionario de datos, order_id, product_id y quantity deben ser enteros, y
unit_price y discount deben ser decimales. Como los datos se cargaron como texto, se
intenta convertir cada columna a número y se listan los valores que no se pueden
convertir (sin contar los vacíos, que ya se cuantificaron en la sección de faltantes).


In [10]:
def revisar_numerica(df, columna, entero=False):
  serie = df[columna].str.strip()
  vacios = (serie == "").sum()
  convertido = pd.to_numeric(serie, errors="coerce")
  no_numericos = serie[convertido.isna() & (serie != "")]

  print(f"--- {columna} ---")
  print("Convertibles a número:", convertido.notna().sum())
  print("Vacíos:", vacios)
  print("No numéricos:", len(no_numericos))
  if len(no_numericos) > 0:
    display(no_numericos.value_counts())

  if entero:
    con_decimales = serie[convertido.notna() & (convertido % 1 != 0)]
    print("Con decimales (debería ser entero):", len(con_decimales))
    if len(con_decimales) > 0:
      print(con_decimales.tolist())
  print()



In [11]:
revisar_numerica(df_details, "order_id", entero=True)
revisar_numerica(df_details, "product_id", entero=True)
revisar_numerica(df_details, "unit_price")
revisar_numerica(df_details, "quantity", entero=True)
revisar_numerica(df_details, "discount")

--- order_id ---
Convertibles a número: 2214
Vacíos: 0
No numéricos: 1


,count
order_id,
A10363,1


Con decimales (debería ser entero): 0

--- product_id ---
Convertibles a número: 2200
Vacíos: 15
No numéricos: 0
Con decimales (debería ser entero): 0

--- unit_price ---
Convertibles a número: 2162
Vacíos: 25
No numéricos: 28


,count
unit_price,
N/A,8
free,6
$12,1
21.05 USD,1
"30,4",1
"123,79",1
$27.8,1
$30.4,1
7.75 USD,1



--- quantity ---
Convertibles a número: 2192
Vacíos: 20
No numéricos: 3


,count
quantity,
diez,3


Con decimales (debería ser entero): 3
['42.5', '2.5', '15.5']

--- discount ---
Convertibles a número: 2175
Vacíos: 31
No numéricos: 9


,count
discount,
"0,05",4
15%,3
cinco,2


In [12]:
revisar_numerica(df_orders, "order_id", entero=True)
revisar_numerica(df_orders, "employee_id", entero=True)
revisar_numerica(df_orders, "ship_via", entero=True)
revisar_numerica(df_orders, "freight")

--- order_id ---
Convertibles a número: 865
Vacíos: 0
No numéricos: 0
Con decimales (debería ser entero): 0

--- employee_id ---
Convertibles a número: 853
Vacíos: 12
No numéricos: 0
Con decimales (debería ser entero): 0

--- ship_via ---
Convertibles a número: 865
Vacíos: 0
No numéricos: 0
Con decimales (debería ser entero): 0

--- freight ---
Convertibles a número: 822
Vacíos: 27
No numéricos: 16


,count
freight,
$89,1
"176,48",1
"9,26",1
$35.03,1
"110,37",1
$4.2,1
"45,13",1
"38,06",1
"22,21",1


In [13]:
for col in ["order_date", "required_date", "shipped_date"]:
  s = df_orders[col].str.strip()
  iso = s.str.fullmatch(r"\d{4}-\d{2}-\d{2}")
  print(f"--- {col} ---")
  print("Con formato AAAA-MM-DD:", iso.sum())
  print("Vacíos:", (s == "").sum())
  print("Con otro formato:", (~iso & (s !="")).sum())
  print()

--- order_date ---
Con formato AAAA-MM-DD: 790
Vacíos: 11
Con otro formato: 64

--- required_date ---
Con formato AAAA-MM-DD: 794
Vacíos: 11
Con otro formato: 60

--- shipped_date ---
Con formato AAAA-MM-DD: 781
Vacíos: 23
Con otro formato: 61



**Conclusión:** En order_details se encontraron 41 valores que no se pueden convertir al tipo esperado: 28 en unit_price (N/A, free, símbolos de moneda y comas decimales), 9 en discount (comas decimales, porcentajes y palabras), 3 en quantity (texto) y 1 en order_id(A10363); además, 3 cantidades tienen decimales. En orders, freight tiene 16 valores no numéricos (comas decimales y símbolos de moneda) y las tres columnas de fecha tienen entre 60 y 64 valores que no siguen el formato AAAA-MM-DD. Las columnas emplyee_id y ship_via no presentan problemas de tipo, y order_id de orders tampoco.

###Formatos inconsistentes

Se revisan los valores que representan lo mismo pero están escritos de distinta forma: espacios sobrantes, mayúsculas y minúsculas, nombres de país y formatos de fecha.

In [14]:
for nombre, df in [("orders", df_orders), ("order_details", df_details)]:
    print(nombre)
    for col in df.columns:
        n = (df[col] != df[col].str.strip()).sum()
        if n > 0:
            print(f"  {col}: {n} valores con espacios al inicio o al final")
    print()

print("order_id con '.0' al final en order_details:",
      df_details["order_id"].str.strip().str.endswith(".0").sum())

orders
  customer_id: 26 valores con espacios al inicio o al final
  ship_name: 27 valores con espacios al inicio o al final
  ship_city: 26 valores con espacios al inicio o al final

order_details
  order_id: 6 valores con espacios al inicio o al final

order_id con '.0' al final en order_details: 3


In [15]:
def variantes_mayusculas(df, columna):
  s = df[columna].str.strip()
  s = s[s != ""]
  return (s.groupby(s.str.lower()).nunique() > 1).sum()

for col in ["customer_id", "ship_name", "ship_address", "ship_city", "ship_region", "ship_country"]:
  print(col, "-> valores escritos de más de una forma:", variantes_mayusculas(df_orders, col))


customer_id -> valores escritos de más de una forma: 0
ship_name -> valores escritos de más de una forma: 21
ship_address -> valores escritos de más de una forma: 8
ship_city -> valores escritos de más de una forma: 14
ship_region -> valores escritos de más de una forma: 0
ship_country -> valores escritos de más de una forma: 10


In [16]:
paises_validos = ["USA", "Germany", "Brazil", "France", "UK", "Venezuela", "Austria",
                  "Sweden", "Italy", "Canada", "Mexico", "Finland", "Ireland", "Spain",
                  "Denmark", "Switzerland", "Belgium", "Argentina", "Portugal", "Poland", "Norway"]

pais = df_orders["ship_country"].str.strip()
no_estandar = pais[(pais != "") & ~pais.isin(paises_validos)]

print("Valores distintos (incluye el vacío):", pais.nunique())
print("Filas con país estándar:", pais.isin(paises_validos).sum())
print("Filas vacías:", (pais == "").sum())
print("Filas con país no estándar:", len(no_estandar), "| variantes distintas:", no_estandar.nunique())
display(no_estandar.value_counts())

Valores distintos (incluye el vacío): 50
Filas con país estándar: 789
Filas vacías: 10
Filas con país no estándar: 66 | variantes distintas: 28


,count
ship_country,
United States,6
france,5
GERMANY,5
Germnay,5
brazil,4
México,3
uk,3
Frnace,3
Italia,2


In [17]:
import re

nombres = {
    "9999-99-99": "AAAA-MM-DD (correcto)",
    "99/99/9999": "DD/MM/AAAA o MM/DD/AAAA",
    "99-99-9999": "DD-MM-AAAA o MM-DD-AAAA",
    "99.99.99": "DD.MM.AA",
    "99999999": "AAAAMMDD",
    "9999/99/99": "AAAA/MM/DD",
    "9 A A A 9999": "D de mes de AAAA",
    "99 A A A 9999": "D de mes de AAAA",
    "A 99, 9999": "Mes DD, AAAA",
}

def formato_fecha(x):
    x = x.strip()
    if x == "":
        return "vacío"
    patron = re.sub(r"[A-Za-zÁ-ú]+", "A", re.sub(r"\d", "9", x))
    return nombres.get(patron, "otro")

columnas_fecha = ["order_date", "required_date", "shipped_date"]
tabla = pd.DataFrame({c: df_orders[c].map(formato_fecha).value_counts() for c in columnas_fecha})
display(tabla.fillna(0).astype(int))

,order_date,required_date,shipped_date
AAAA-MM-DD (correcto),790,794,781
AAAA/MM/DD,9,3,13
AAAAMMDD,9,7,11
D de mes de AAAA,13,11,5
DD-MM-AAAA o MM-DD-AAAA,8,10,8
DD.MM.AA,10,3,11
DD/MM/AAAA o MM/DD/AAAA,10,11,9
"Mes DD, AAAA",5,15,4
vacío,11,11,23


In [18]:
total = ambiguas = dia_primero = mes_primero = 0
for col in columnas_fecha:
    s = df_orders[col].str.strip()
    m = s[s.str.fullmatch(r"\d{2}[/-]\d{2}[/-]\d{4}")]
    a = m.str[:2].astype(int)
    b = m.str[3:5].astype(int)
    total += len(m)
    ambiguas += ((a <= 12) & (b <= 12)).sum()
    dia_primero += (a > 12).sum()
    mes_primero += (b > 12).sum()

print("Fechas con barra o guion y año de 4 dígitos:", total)
print("Ambiguas (día y mes ≤ 12):", ambiguas)
print("Con el día primero (primer número > 12):", dia_primero)
print("Con el mes primero (segundo número > 12):", mes_primero)

Fechas con barra o guion y año de 4 dígitos: 56
Ambiguas (día y mes ≤ 12): 19
Con el día primero (primer número > 12): 20
Con el mes primero (segundo número > 12): 17


**Conclusión.** En orders se encontraron espacios sobrantes en customer_id (26),
ship_name (27) y ship_city (26), y en order_details en order_id (6, más 3 con ".0" al
final). Varios campos de texto se escriben de más de una forma por diferencias de
mayúsculas (ship_name 21, ship_city 14, ship_country 10, ship_address 8). En ship_country
hay 28 variantes no estándar (66 filas) frente a una lista de 21 países válidos:
mayúsculas, errores de escritura, otro idioma y siglas. Las tres columnas de fecha usan 8
formatos distintos; en las que traen barra o guion con año de 4 dígitos conviven las
convenciones día/mes y mes/día (20 y 17 casos), y 19 de ellas son ambiguas.

###Valores fuera de rango

Se revisan valores que tienen el tipo correcto pero que no son posibles según el diccionario de datos: cantidades no positivas, descuentos no permitidos, fletes negativos y fechas incoherentes entre sí.

In [19]:
def a_numero(serie):
    return pd.to_numeric(serie.str.strip(), errors="coerce")

q = a_numero(df_details["quantity"])
p = a_numero(df_details["unit_price"])
desc = a_numero(df_details["discount"])

print("quantity ≤ 0:", (q <= 0).sum(), sorted(q[q <= 0].tolist()))
print("quantity > 130:", (q > 130).sum(), q[q > 130].tolist())
print("unit_price ≤ 0:", (p <= 0).sum())

descuentos_validos = [0, 0.05, 0.10, 0.15, 0.20, 0.25]
desc_invalido = desc[desc.notna() & ~desc.isin(descuentos_validos)]
print("discount fuera de los valores permitidos:", len(desc_invalido))
display(desc_invalido.value_counts())

quantity ≤ 0: 5 [-25.0, -24.0, -5.0, 0.0, 0.0]
quantity > 130: 1 [999.0]
unit_price ≤ 0: 0
discount fuera de los valores permitidos: 15


,count
discount,
-0.10,3
0.03,3
1.50,3
0.02,2
0.04,2
0.01,1
0.06,1


In [20]:
f = a_numero(df_orders["freight"])
e = a_numero(df_orders["employee_id"])
v = a_numero(df_orders["ship_via"])

print("freight < 0:", (f < 0).sum(), sorted(f[f < 0].tolist()))
print("freight máximo:", f.max())
print("employee_id fuera de 1–9:", ((e < 1) | (e > 9)).sum())
print("ship_via fuera de {1, 2, 3}:", (~v.dropna().isin([1, 2, 3])).sum())

freight < 0: 4 [-136.54, -24.69, -12.36, -10.22]
freight máximo: 1007.64
employee_id fuera de 1–9: 0
ship_via fuera de {1, 2, 3}: 0


In [21]:
def fecha_iso(col):
    s = df_orders[col].str.strip()
    return pd.to_datetime(s.where(s.str.fullmatch(r"\d{4}-\d{2}-\d{2}")),
                          format="%Y-%m-%d", errors="coerce")

pedido, requerido, enviado = fecha_iso("order_date"), fecha_iso("required_date"), fecha_iso("shipped_date")

print("Rango de order_date:", pedido.min().date(), "a", pedido.max().date())

ok = pedido.notna() & enviado.notna()
print("Enviado antes de la fecha del pedido:", (enviado[ok] < pedido[ok]).sum(), "de", ok.sum())
ok = pedido.notna() & requerido.notna()
print("Requerido antes de la fecha del pedido:", (requerido[ok] < pedido[ok]).sum(), "de", ok.sum())
ok = requerido.notna() & enviado.notna()
print("Enviado después de la fecha requerida:", (enviado[ok] > requerido[ok]).sum(), "de", ok.sum())

Rango de order_date: 1996-07-04 a 1998-05-06
Enviado antes de la fecha del pedido: 0 de 713
Requerido antes de la fecha del pedido: 0 de 730
Enviado después de la fecha requerida: 35 de 713


**Conclusión.** En order_details se encontraron 5 cantidades no positivas (-25, -24, -5,
0 y 0), 1 cantidad atípica (999) y 15 descuentos fuera de los valores permitidos (-0.1,
0.01, 0.02, 0.03, 0.04, 0.06 y 1.5); ningún precio es menor o igual a cero. En orders hay
4 fletes negativos, y employee_id y ship_via están dentro de los rangos esperados. Entre
las fechas con formato AAAA-MM-DD no hay incoherencias de orden: ningún pedido se envió
ni se requirió antes de hacerse. Los 35 envíos posteriores a la fecha requerida son una
observación de negocio y no un error de calidad.

### Integridad entre tablas

Se comprueba que cada order_id de order_details exista en orders, y que cada pedido de
orders tenga al menos una línea en order_details. Primero se compara con los valores tal
como vienen y después quitando los espacios y el ".0" final.

In [22]:
ids_orders = set(df_orders["order_id"].str.strip())
ids_details = set(df_details["order_id"])

print("Pedidos distintos en orders:", len(ids_orders))
print("order_id distintos en order_details:", len(ids_details))
print("order_id de order_details que no existen en orders:", len(ids_details - ids_orders))
print("Pedidos de orders sin líneas en order_details:", len(ids_orders - ids_details),
      sorted(ids_orders - ids_details))

Pedidos distintos en orders: 830
order_id distintos en order_details: 842
order_id de order_details que no existen en orders: 15
Pedidos de orders sin líneas en order_details: 3 ['10422', '10545', '11061']


In [23]:
ids_norm = df_details["order_id"].str.strip().str.replace(r"\.0$", "", regex=True)
huerfanos = ids_norm[~ids_norm.isin(ids_orders)]

print("Tras quitar espacios y '.0':")
print("order_id de order_details sin pedido en orders:", huerfanos.nunique(), sorted(huerfanos.unique()))
print("Líneas afectadas:", len(huerfanos))
print("Pedidos de orders sin líneas:", len(ids_orders - set(ids_norm)))

Tras quitar espacios y '.0':
order_id de order_details sin pedido en orders: 6 ['19473', '19475', '19484', '19550', '19614', 'A10363']
Líneas afectadas: 6
Pedidos de orders sin líneas: 0


**Conclusión.** Al comparar los order_id tal como vienen, 15 de order_details no existen
en orders y 3 pedidos de orders aparecen sin líneas. La mayor parte se debe a 9
identificadores con espacios o ".0" al final: una vez normalizados, los 830 pedidos
tienen al menos una línea y quedan 6 líneas sin pedido (19473, 19475, 19484, 19550,
19614 y A10363). Parecen errores de captura de pedidos que sí existen (10473, 10475,
10484, 10550, 10614 y 10363), pero con los datos disponibles no se puede confirmar.

### Caracteres mal codificados

Se buscan en las columnas de texto caracteres que no pertenecen al texto original: las
secuencias "Ã" y "Â" que aparecen cuando una letra con acento se lee con la codificación
equivocada, y el carácter de reemplazo "�". Las letras con acento legítimas (é, ö, ñ...)
no se cuentan.

In [24]:
patron = r"[ÃÂ�]"

for nombre, df in [("orders", df_orders), ("order_details", df_details)]:
    total = 0
    print(nombre)
    for col in df.columns:
        n = df[col].str.contains(patron, regex=True).sum()
        if n > 0:
            total += n
            print(f"  {col}: {n} valores con caracteres extraños")
    print("  Total de celdas afectadas:", total)
    print()

orders
  ship_name: 20 valores con caracteres extraños
  ship_address: 8 valores con caracteres extraños
  ship_city: 11 valores con caracteres extraños
  ship_postal_code: 7 valores con caracteres extraños
  Total de celdas afectadas: 46

order_details
  Total de celdas afectadas: 0



In [25]:
cols_texto = ["ship_name", "ship_address", "ship_city", "ship_postal_code"]
afectadas = df_orders[df_orders[cols_texto].apply(lambda c: c.str.contains(patron, regex=True)).any(axis=1)]

print("Filas afectadas:", len(afectadas))
print("Con el carácter �:", sum(df_orders[c].str.contains("�").sum() for c in df_orders.columns))
print("Con secuencias tipo 'Ã¼', 'Ã±', 'Ã©':", sum(df_orders[c].str.contains("[ÃÂ]", regex=True).sum() for c in df_orders.columns))
display(afectadas[["order_id"] + cols_texto].head(10))

Filas afectadas: 46
Con el carácter �: 23
Con secuencias tipo 'Ã¼', 'Ã±', 'Ã©': 23


,order_id,ship_name,ship_address,ship_city,ship_postal_code
5,10915,Tortuga Restaurante,Avda. Azteca 123,México D.F.,05Ã¼033
17,10904,White Clover Markets,1029 - 12th Ave. S.,SeÃ¼attle,98124
75,10565,Mère Paillarde,43 rue St. Laurent,Montréal,H�1J 1C3
84,10328,Furia Bacalhau e Frutos do Mar,Jardim das rosas nÃ©. 32,,1675
86,10606,�Tradiçao Hipermercados,"Av. Inês de Castro, 414",Sao Paulo,05634-030
92,10443,Reggiani CaseificÃ©i,Strada Provinciale 124,Reggio Emilia,42100
118,10680,Old World Delicatessen,2743 Bering St.,Anc�horage,99508
127,10349,Split Rail Beer & Ale,P.O. Box 555,LaÃ¼nder,82520
188,10494,Comércio Mineiro,"Av. dos �Lusíadas, 23",Sao Paulo,05432-043
222,10973,La corne d'abondancÃ±e,"67, avenue de l''Europe",Versailles,78000


In [26]:
ejemplos = df_orders.loc[df_orders["ship_city"].str.contains("Ã", regex=False), "ship_city"].unique()
for x in ejemplos:
    print(x, "→", x.encode("latin-1").decode("utf-8"))

SeÃ¼attle → Seüattle
LaÃ¼nder → Laünder
TsawaÃ±ssen → Tsawañssen


**Conclusión.** En orders se encontraron 46 celdas con caracteres extraños, cada una en
una fila distinta: ship_name (20), ship_city (11), ship_address (8) y ship_postal_code
(7); order_details no tiene casos. La mitad contiene el carácter de reemplazo "�" y la
otra mitad secuencias como "Ã¼", "Ã±" o "Ã©". La reparación habitual de mojibake no
recupera el texto original (por ejemplo, "SeÃ¼attle" pasa a "Seüattle" y no a
"Seattle"), lo que sugiere que los caracteres fueron insertados dentro de las palabras y
que la limpieza deberá eliminarlos.

## Resumen general del diagnóstico

| Categoría | `orders` | `order_details` |
|---|---|---|
| **Valores faltantes** | `ship_region` 525 (esperables), `freight` 27, `shipped_date` 23, `ship_postal_code` 21, `customer_id` 16, entre otras | `discount` 31, `unit_price` 25, `quantity` 20, `product_id` 15 |
| **Duplicados** | 25 filas idénticas; 10 pedidos repetidos con `ship_name` en otras mayúsculas | 60 filas idénticas; 0 repetidas por `order_id` + `product_id` |
| **Tipos incorrectos** | `freight`: 16 no numéricos; fechas fuera de `AAAA-MM-DD`: 64 (`order_date`), 60 (`required_date`) y 61 (`shipped_date`) | `unit_price` 28, `discount` 9, `quantity` 3 (más 3 con decimales), `order_id` 1 |
| **Formatos inconsistentes** | 8 formatos de fecha (19 de las 56 fechas con barra o guion son ambiguas); 66 filas con país no estándar (28 variantes); espacios en `customer_id` (26), `ship_name` (27) y `ship_city` (26); mayúsculas inconsistentes en `ship_name`, `ship_city`, `ship_country` y `ship_address` | `order_id` con espacios (6) o con `.0` (3) |
| **Valores fuera de rango** | 4 fletes negativos | 5 cantidades no positivas, 1 atípica (999) y 15 descuentos no permitidos |
| **Integridad entre tablas** | 3 pedidos sin líneas (0 tras normalizar los `order_id`) | 15 `order_id` sin pedido (6 tras normalizar: 19473, 19475, 19484, 19550, 19614 y A10363) |
| **Encoding** | 46 celdas con caracteres insertados: `ship_name` 20, `ship_city` 11, `ship_address` 8, `ship_postal_code` 7 | 0 |

*Las cifras corresponden a los datos originales, antes de la limpieza, y un mismo registro puede aparecer en más de una categoría.*